In [1]:
import pandas as pd

df = pd.read_csv('speeches_processed.csv')

In [2]:
n_words = df["speech_en"].astype(str).str.split().str.len()
print(n_words.describe(percentiles=[0.5, 0.75, 0.9, 0.99]).round(0))
for limite in [200, 280, 500, 1500]:
    print(f"Acima de {limite} palavras: {(n_words > limite).mean():.1%}")

count    563696.0
mean        188.0
std         193.0
min           1.0
50%         154.0
75%         205.0
90%         338.0
99%         937.0
max        8548.0
Name: speech, dtype: float64
Acima de 200 palavras: 28.6%
Acima de 280 palavras: 14.4%
Acima de 500 palavras: 4.6%
Acima de 1500 palavras: 0.3%


In [3]:
DATE_COL = "date"

df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df["year"] = df[DATE_COL].dt.year

BENCH_YEARS = [2015, 2021]
df_bench = df[df["year"].isin(BENCH_YEARS)].copy().reset_index(drop=True)

In [5]:
df_bench.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92678 entries, 0 to 92677
Data columns (total 34 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   doc_id                      92678 non-null  float64       
 1   file                        92678 non-null  object        
 2   term_no                     92678 non-null  int64         
 3   date                        92678 non-null  datetime64[ns]
 4   datemonth                   92678 non-null  float64       
 5   year                        92678 non-null  int32         
 6   agenda                      92678 non-null  object        
 7   firstname                   92678 non-null  object        
 8   lastname                    92678 non-null  object        
 9   country                     92678 non-null  object        
 10  language                    92678 non-null  object        
 11  nationality                 92678 non-null  object    

In [15]:
import json, os
COLS = ["doc_id", 
        "file", 
        "term_no", 
        "date", 
        "datemonth", 
        "year", 
        "agenda", 
        "firstname", 
        "lastname", 
        "country", 
        "language", 
        "nationality", 
        "gender", 
        "birth_year", 
        "birth_place", 
        "mepid", 
        "moderator", 
        "party_db_code", 
        "epg_short", 
        "epg_long", 
        "epg_partyfacts_id", 
        "party_name", 
        "partyfacts_id", 
        "ches_id", 
        "gps_id", 
        "speech", 
        "speech_en", 
        "agenda_clean", 
        "agenda_en", 
        "speech_no_stopwords", 
        "ano", 
        "speech_no_custom_stopwords", 
        "speech_lemmatized",
         "speech_stemmed"]
splits = json.load(open("lda_bayes/splits.json"))
split_of = {i: nome for nome, ids in splits.items() for i in ids}

bench = df_bench.loc[list(split_of), COLS].copy()
# listas de tokens -> texto (o CSV não guarda listas); to_text é a função do LDA
bench["speech_no_custom_stopwords"] = bench["speech_no_custom_stopwords"].apply(to_text)
bench["split"] = bench.index.map(split_of)
bench["doc_idx"] = bench.index
os.makedirs("data", exist_ok=True)
bench.to_csv("data/bench_lda.csv", index=False)
print(bench["split"].value_counts())

NameError: name 'to_text' is not defined

In [ ]:
# %% Configuração
from pathlib import Path

# Ficheiro exportado no Passo 0: os mesmos discursos e o mesmo split do benchmark LDA.
DATA_PATH = Path("data/bench_lda.parquet")
TEXT_COL = "speech"            # texto ORIGINAL: sem stop words removidas, sem lemma, sem stem
ID_COL = "doc_idx"             # índice do df_bench no LDA (permite cruzar resultados)
DATE_COL = "date"
PARTY_COL = "political_group"
# "split" separa treino/validação/teste. A variante base do LDA vai servir na parte 2
# para as palavras dos tópicos e para calcular o c_v sobre os mesmos tokens do LDA.
EXTRA_COLS = ["split", "speech_no_custom_stopwords"]
YEARS = None                   # None = não filtrar (o ficheiro já só tem os 2 anos)
MIN_DOC_WORDS = 0              # 0 = não filtrar (o MIN_TOKENS = 20 do LDA já foi aplicado)

# Unidade de documento:
#   CHUNK = False -> cada intervenção inteira é um documento (recomendado para os teus dados:
#                    mediana 154 palavras, só 0,3% acima de 1500)
#   CHUNK = True  -> intervenções longas são divididas em excertos de ~MAX_WORDS palavras
CHUNK = False
MAX_WORDS = 200                # só usado com CHUNK = True
MIN_WORDS = 40                 # só usado com CHUNK = True

# Todos os modelos leem no máximo este número de tokens, para verem o mesmo texto.
# 2048 é o limite do EmbeddingGemma (~1500 palavras).
MAX_SEQ_TOKENS = 2048

OUT_DIR = Path("bertopic_bench")
CHUNKS_PATH = OUT_DIR / "chunks.parquet"   # uma linha por documento (intervenção ou excerto)
EMB_DIR = OUT_DIR / "embeddings"

# O encode processa primeiro os textos mais longos: se faltar memória, o erro aparece logo
# no início. Nesse caso baixa para 8-16.
BATCH_SIZE = 32
BLOCK_SIZE = 20_000            # guarda em blocos: se o PC desligar, retoma onde ficou
SAMPLE_FOR_TIMING = 256

MODELS = {
    "mpnet": {
        "name": "sentence-transformers/all-mpnet-base-v2",
        "prompt": None,
    },
    "gte_modernbert": {
        "name": "Alibaba-NLP/gte-modernbert-base",
        "prompt": None,
    },
    "embeddinggemma": {
        "name": "google/embeddinggemma-300m",
        "prompt": "task: clustering | query: ",
        "allow_fp16": False,   # o modelo não suporta float16
    },
    "qwen3_06b": {
        "name": "Qwen/Qwen3-Embedding-0.6B",
        "prompt": "Instruct: Identify the main topic of this parliamentary speech\nQuery: ",
    },
}

# O mpnet só lê 384 tokens (~280 palavras): sem divisão, cortaria ~14% dos teus discursos.
# Junta mais modelos aqui se algum for demasiado lento no teu PC, ex.: "qwen3_06b".
SKIP = [] if CHUNK else ["mpnet"]
RUN_MODELS = [k for k in MODELS if k not in SKIP]

# %% Login no Hugging Face (só para o EmbeddingGemma, que exige aceitar a licença)
# 1) Aceita a licença em https://huggingface.co/google/embeddinggemma-300m
# 2) Descomenta e corre uma vez (pede um token de leitura):
# from huggingface_hub import login; login()

# %% Imports e dispositivo
import gc
import re
import time

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | torch {torch.__version__} | modelos: {RUN_MODELS}")
EMB_DIR.mkdir(parents=True, exist_ok=True)

# %% Carregar os discursos do benchmark
if DATA_PATH.suffix == ".csv":
    df = pd.read_csv(DATA_PATH)
else:
    df = pd.read_parquet(DATA_PATH)

df[DATE_COL] = pd.to_datetime(df[DATE_COL])
if YEARS:
    df = df[df[DATE_COL].dt.year.isin(YEARS)].copy()
if ID_COL not in df.columns:
    df[ID_COL] = df.index
df = df.dropna(subset=[TEXT_COL])
df[TEXT_COL] = df[TEXT_COL].astype(str)

n_words = df[TEXT_COL].str.split().str.len()
if MIN_DOC_WORDS > 0:
    df = df[n_words >= MIN_DOC_WORDS].copy()
    n_words = n_words.loc[df.index]

print(f"{len(df):,} discursos | anos {sorted(df[DATE_COL].dt.year.unique().tolist())}")
if "split" in df.columns:
    print(df["split"].value_counts().to_string())
print(n_words.describe(percentiles=[0.5, 0.75, 0.9, 0.99]).round(0).to_string())
for limite in [200, 280, 500, 1500]:
    print(f"Acima de {limite} palavras: {(n_words > limite).mean():.1%}")

# %% Preparar os documentos
_SENT_SPLIT = re.compile(r"(?<=[.!?;])\s+")


def normalize(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def split_sentences(text: str) -> list[str]:
    return [s for s in _SENT_SPLIT.split(normalize(text)) if s]


def chunk_text(text: str, max_words: int = MAX_WORDS, min_words: int = MIN_WORDS) -> list[str]:
    chunks, current = [], []
    for sent in split_sentences(text):
        words = sent.split()
        # frase sozinha maior do que o limite: corta à força
        while len(words) > max_words:
            if current:
                chunks.append(" ".join(current))
                current = []
            chunks.append(" ".join(words[:max_words]))
            words = words[max_words:]
        if current and len(current) + len(words) > max_words:
            chunks.append(" ".join(current))
            current = []
        current.extend(words)
    if current:
        chunks.append(" ".join(current))
    # último excerto muito curto vai para o anterior (fica no máximo com max_words + min_words)
    if len(chunks) > 1 and len(chunks[-1].split()) < min_words:
        last = chunks.pop()
        chunks[-1] = f"{chunks[-1]} {last}"
    return chunks


keep_cols = [ID_COL, DATE_COL, PARTY_COL, *EXTRA_COLS]
rows = []
for meta, text in zip(df[keep_cols].itertuples(index=False), df[TEXT_COL]):
    pieces = chunk_text(text) if CHUNK else [normalize(text)]
    for i, piece in enumerate(pieces):
        if piece:
            rows.append((*meta, i, piece))

chunks = pd.DataFrame(rows, columns=[*keep_cols, "chunk_idx", "text"])
chunks["n_words"] = chunks["text"].str.split().str.len()
chunks.to_parquet(CHUNKS_PATH, index=False)
unidade = "excertos" if CHUNK else "documentos (1 por intervenção)"
print(f"{len(chunks):,} {unidade} de {chunks[ID_COL].nunique():,} discursos -> {CHUNKS_PATH}")

# %% Verificação: quantos documentos seriam truncados
from transformers import AutoTokenizer

check_key = "mpnet" if CHUNK else RUN_MODELS[0]
token_limit = 384 if CHUNK else MAX_SEQ_TOKENS
tok = AutoTokenizer.from_pretrained(MODELS[check_key]["name"])
sample = chunks["text"].sample(min(5000, len(chunks)), random_state=42)
n_tok = sample.map(lambda t: len(tok(t, verbose=False)["input_ids"]))
print(f"Tokens por documento (amostra, tokenizador {check_key}): "
      f"mediana {n_tok.median():.0f}, p99 {n_tok.quantile(0.99):.0f}, máx {n_tok.max()}")
print(f"Acima de {token_limit} tokens (seriam truncados): {(n_tok > token_limit).mean():.2%}")

# %% Funções de encoding
def load_model(key: str) -> SentenceTransformer:
    cfg = MODELS[key]
    model = SentenceTransformer(cfg["name"], device=DEVICE)
    model.max_seq_length = min(model.max_seq_length, MAX_SEQ_TOKENS)
    if DEVICE == "cuda" and cfg.get("allow_fp16", True):
        model.half()
    return model


def encode(model: SentenceTransformer, texts: list[str], prompt, progress: bool = True) -> np.ndarray:
    emb = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        prompt=prompt,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=progress,
    )
    return emb.astype(np.float32)


def free_memory():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


texts_all = chunks["text"].tolist()

# %% Estimar o tempo de cada modelo antes de correr tudo
sample_texts = chunks["text"].sample(min(SAMPLE_FOR_TIMING, len(chunks)), random_state=0).tolist()
for key in RUN_MODELS:
    model = load_model(key)
    encode(model, sample_texts[:16], MODELS[key]["prompt"], progress=False)  # aquecimento
    t0 = time.perf_counter()
    encode(model, sample_texts, MODELS[key]["prompt"], progress=False)
    minutes = (time.perf_counter() - t0) / len(sample_texts) * len(texts_all) / 60
    print(f"{key:15s} ~{minutes:,.0f} min para {len(texts_all):,} documentos")
    del model
    free_memory()

# %% Calcular e guardar os embeddings (retoma blocos já feitos)
for key in RUN_MODELS:
    final_path = EMB_DIR / f"{key}.npy"
    if final_path.exists():
        print(f"{key}: já existe, a saltar")
        continue

    model = None
    n_blocks = (len(texts_all) + BLOCK_SIZE - 1) // BLOCK_SIZE
    part_paths = []
    for b in range(n_blocks):
        part_path = EMB_DIR / f"{key}_part{b:03d}.npy"
        part_paths.append(part_path)
        if part_path.exists():
            continue
        if model is None:
            model = load_model(key)
        block = texts_all[b * BLOCK_SIZE:(b + 1) * BLOCK_SIZE]
        print(f"{key}: bloco {b + 1}/{n_blocks}")
        np.save(part_path, encode(model, block, MODELS[key]["prompt"]))

    emb = np.concatenate([np.load(p) for p in part_paths])
    np.save(final_path, emb)
    for p in part_paths:
        p.unlink()
    print(f"{key}: {emb.shape} -> {final_path}")
    del model
    free_memory()

# %% Verificação final
chunks = pd.read_parquet(CHUNKS_PATH)
for key in RUN_MODELS:
    emb = np.load(EMB_DIR / f"{key}.npy")
    norms = np.linalg.norm(emb, axis=1)
    ok = emb.shape[0] == len(chunks) and not np.isnan(emb).any()
    print(f"{key:15s} {str(emb.shape):16s} normas {norms.min():.3f}-{norms.max():.3f}  {'OK' if ok else 'PROBLEMA'}")

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.